# H&M 검증된 거래 원본을 Runpod으로 옮기기

**GPU를 사용하지 않습니다.** Colab 런타임 유형을 CPU로 두고 학교 Drive가 있는 계정으로 연결하세요. 잘못 변환된 로컬 Parquet나 과거 분할ZIP은 사용하지 않습니다. 원본은 읽기만 하며 삭제·수정하지 않습니다.

준비 패키지의 `hm_support.zip`에는 검증된 상품 파일·참조2개가 있습니다. 여기서는 거래 원본 하나를 해시 확인 후 내보냅니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, zipfile, shutil
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
SOURCE = Path('/content/drive/MyDrive/논문/data/raw/hm/transactions_train.parquet')
EXPECTED = '0d90884fc1bc400161e6aeba238cba94183998bedfeb4ebbf4962e17d0c8489b'
def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(8*1024**2), b''): h.update(b)
    return h.hexdigest()
assert SOURCE.is_file(), '학교 Drive의 원본 경로를 확인하세요.'
assert sha(SOURCE) == EXPECTED, '검증된 원본이 아닙니다. CSV 재변환이나 과거 ZIP 사용 금지.'
assert shutil.disk_usage('/content').free > SOURCE.stat().st_size * 2 + 1024**3
print('원본 SHA256 확인 완료:', SOURCE.stat().st_size, 'bytes')


In [ ]:
OUTPUT = Path('/content/hm_raw_verified.zip')
NAME = 'raw/hm/transactions_train.parquet'
with zipfile.ZipFile(OUTPUT, 'w', zipfile.ZIP_STORED) as z:
    z.writestr('manifest.json', json.dumps({NAME: EXPECTED}))
    z.write(SOURCE, NAME)
with zipfile.ZipFile(OUTPUT) as z:
    h = hashlib.sha256()
    with z.open(NAME) as f:
        for b in iter(lambda: f.read(8*1024**2), b''): h.update(b)
    assert h.hexdigest() == EXPECTED
print('원본 바이트와 ZIP 내부 파일 일치:', OUTPUT)
from google.colab import files
files.download(str(OUTPUT))


다운로드한 `hm_raw_verified.zip`을 Runpod JupyterLab의 `/workspace/runpod_start`에 업로드하고 시작 노트북 2번 셀부터 실행하세요. 이 내보내기 작업은 학습·평가를 하지 않습니다.